# Decoder Pipeline - 3D U-Net vs V-Net  (HPC full-training twin)

Full-resolution (`256^3`) training of the **3D U-Net** and **V-Net** decoders on the Sunway HPC GPU.
This is the twin of `notebooks/modeling/decoder_pipeline.ipynb`; the **only** differences are in the
CONFIG cell (CUDA device, 256^3 target, full epochs, mixed precision, gradient checkpointing, data
workers). Verify the pipeline locally first, then run this for the real comparison.

Pipeline: `AP+LAT DRRs -> shared encoder/fusion -> 3D features [64,128,256,512] -> U-Net/V-Net decoder -> occupancy volume`

### How to run

0. **First** run `frontend_pretrain.ipynb` for the **same `FOLD`** so `models/front_end_fold{FOLD}.pth`
   and `models/decoders/decoder_split_fold{FOLD}.csv` exist. With `REGIME="frozen"` this notebook
   loads and **freezes** that whole front-end, so only the decoder trains. (If the file is missing it
   falls back to a frozen SimCLR encoder + trainable fusion/lift and prints a warning.)
1. Run the cells top to bottom. The **CONFIG** cell is the only place you change settings.
2. **The comparison is a grid.** For each `FOLD` in `0..N_FOLDS-1`, and each `REGIME` in
   `{"frozen", "finetuned"}`, run the whole notebook with `MODEL="unet"`, then `MODEL="vnet"`. Each
   run writes its own `models/decoders/fold{FOLD}/{REGIME}/{MODEL}/` folder and a per-knee metric CSV.
3. When the grid is done, open `decoder_comparison.ipynb` to pool the per-knee CSVs and run the
   paired U-Net vs V-Net Wilcoxon test (overall and within the fractured subgroup).

In [ ]:
import os, math, random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ============================= CONFIG (HPC - full 256^3 training) =============================
# Mirrors the local notebook; only these knobs differ. Run on the Sunway HPC GPU.
ENV          = "HPC"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL        = "unet"        # train "unet", then re-run with "vnet" for the comparison
TARGET_RES   = 256
LIFT_DEPTH   = 16
EPOCHS       = 40
BATCH_SIZE   = 1             # 256^3 is memory-heavy; raise to 2 only if the GPU allows
LR           = 1e-4
CKPT_EVERY   = 5
USE_AMP      = True
USE_GRAD_CKPT = True
NUM_WORKERS  = 4
GT_THRESH    = 0.40
INCLUDE_GEOMETRIC = False
DEEP_SUPERVISION  = False
# --- cross-validation (knee-level, dataset-stratified) ---
N_FOLDS      = 5             # k-fold CV over knees (matches FracReconNet); every knee is tested once
FOLD         = 0             # which fold is held out as TEST this run (0..N_FOLDS-1)
# --- front-end regime (run BOTH and compare in decoder_comparison.ipynb) ---
REGIME       = "frozen"      # "frozen": strict decoder isolation (front-end frozen -> byte-identical
                             #   features for U-Net & V-Net). "finetuned": train encoder+fusion+lift
                             #   +decoder jointly (realistic capacity; NOT a pure decoder ablation).
FREEZE_FRONTEND = (REGIME == "frozen")   # derived from REGIME so the two regimes stay consistent
FREEZE_ENCODER  = (REGIME == "frozen")   # finetuned regime trains the SimCLR backbone too
PRETRAINED   = True
SMOKE_TEST   = False
SMOKE_CASES_PER_GROUP = 3
RESUME_FROM  = None
EXPLICIT_ROOT = None         # e.g. "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
if DEVICE.type != "cuda":
    print("[warning] CUDA not available - this HPC notebook expects a GPU.")
print("ENV", ENV, "| MODEL", MODEL, "| fold", FOLD, "/", N_FOLDS, "| regime", REGIME,
      "| TARGET_RES", TARGET_RES, "| device", DEVICE, "| epochs", EPOCHS)

In [ ]:
# Resolve the project root robustly (works locally and on HPC, regardless of where
# the notebook is launched from). We look upward for the data/interim/predrr folder,
# which holds the ground-truth CT volumes.
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT           = find_root(Path.cwd())
DATA           = ROOT / "data"
NORMAL_DRR_DIR = DATA / "interim" / "DRRs"                 # AP/LAT DRRs (model inputs)
AUG_DRR_DIR    = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR     = DATA / "interim" / "predrr"               # ground-truth CT volumes
MODELS_DIR     = ROOT / "models"
SIMCLR_CKPT    = MODELS_DIR / "convnextv2_simclr_encoder.pth"      # SimCLR encoder (fold-agnostic; SSL uses no labels)
FRONTEND_CKPT  = MODELS_DIR / ("front_end_fold%d.pth" % FOLD)      # per-fold frozen front-end (frozen regime only)
# per-fold + per-regime + per-model checkpoints, so the 2 models x 2 regimes per fold never collide
CKPT_DIR       = MODELS_DIR / "decoders" / ("fold%d" % FOLD) / REGIME / MODEL
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GT_CACHE_DIR   = DATA / "interim" / ("predrr_occupancy_%d" % TARGET_RES)   # cached binary GT
print("ROOT:", ROOT)
print("checkpoints ->", CKPT_DIR)

## 1. Shared encoder (copied verbatim from `encoder_pipeline.ipynb`)

The encoder is the part both decoders share, so the comparison is fair: **only the decoder
changes**. The code below is copied verbatim from `encoder_pipeline.ipynb` so this notebook is
self-contained — **do not edit it here** (keeping it byte-identical is what lets `front_end.pth`
load with `missing=0 unexpected=0`). When `FREEZE_FRONTEND=True`, `build_model()` loads the
pretrained front-end and **freezes all of it** (encoder + fusion + lift).

What it does, in plain terms:
1. A **ConvNeXtV2** backbone turns each X-ray (AP and LAT) into 4 feature maps at increasing depth.
2. **Hybrid bi-planar fusion** merges the two views: cheap convolution at fine scales (keeps local
   fracture detail), cross-attention at coarse scales (aligns global knee shape).
3. A **2D->3D lift** stacks each fused map into a small 3D feature volume (depth = `LIFT_DEPTH`).

Output: a list of 4 multi-scale 3D feature tensors with channels `[64, 128, 256, 512]` — this is
the *contract* the decoder consumes.

In [ ]:
# ===== Encoder front-end - VERBATIM from encoder_pipeline.ipynb. DO NOT EDIT. =====
# (PRETRAINED / FREEZE_ENCODER are set in the CONFIG cell so they stay visible knobs.)
BACKBONE     = "convnextv2_tiny"
IMG_SIZE     = 256
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types); self.depth = depth
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        self.expand3d = nn.ModuleList([nn.Conv3d(o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)
            f3 = c2d(f2d).unsqueeze(2)
            f3 = F.interpolate(f3, size=(self.depth, H, W), mode="trilinear", align_corners=False)
            fused3d.append(c3d(f3))
        return fused2d, fused3d

print("encoder feature dims:", FEAT_DIMS)

## 2. Data — paired (DRR inputs, binary-occupancy GT)

The (X-ray, CT) pairs are aligned *by construction*: the DRRs were rendered **from** these exact CT
volumes. So for each DRR pair we look up the matching `predrr` CT volume and turn it into the
training target.

**Ground-truth target = binary bone occupancy.** The `predrr` CT is bone-windowed and scaled to
`[0,1]`. We threshold it at `GT_THRESH` to get a `{0,1}` bone mask, resample it to `TARGET_RES^3`
with **nearest-neighbour** (so it stays binary), and cache it to disk (so we threshold once, not
every epoch). Tune `GT_THRESH` using the QA cell near the bottom.

**Splitting** is done at the *knee* level (`dataset, case, side`) so all augmented variants of one
knee land in the same split (no leakage), and healthy/fractured cases are stratified across
train/val/test. We exclude **geometric** augmentations (rotations/flips) by default, because their
3D ground truth would need the same transform applied — photometric variants reuse the base GT.

In [ ]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)
print("paired rows:", len(paired_index),
      "| cases:", paired_index.groupby("dataset")["case"].nunique().to_dict())

# ---- ground truth: predrr CT -> binary bone occupancy at TARGET_RES (cached) ----
def gt_file(dataset, case, side):
    Side = "Right" if str(side).lower().startswith("r") else "Left"
    if dataset == "healthy":
        return PREDRR_DIR / "healthy" / ("%s_%s.nii.gz" % (case, Side))
    return PREDRR_DIR / "fractured" / ("%s_Part%s.nii.gz" % (case, Side))

def load_gt_occupancy(dataset, case, side):
    GT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = GT_CACHE_DIR / ("%s_%s_%s.npy" % (dataset, case, side))
    if cache.exists():
        occ = np.load(cache)
    else:
        vol = nib.load(str(gt_file(dataset, case, side))).get_fdata().astype(np.float32)  # 256^3 [0,1]
        occ = (vol > GT_THRESH).astype(np.float32)
        t = F.interpolate(torch.from_numpy(occ)[None, None], size=(TARGET_RES,) * 3, mode="nearest")
        occ = t[0, 0].numpy().astype(np.float32)
        np.save(cache, occ)
    return torch.from_numpy(occ)[None]   # (1, T, T, T)

class PairedDRRVolumeDataset(Dataset):
    """Returns AP/LAT DRRs (3x256x256) + binary GT occupancy (1,T,T,T) + metadata."""
    def __init__(self, df, transform=paired_tf):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ap = self.transform(load_drr(r.ap)); lat = self.transform(load_drr(r.lat))
        gt = load_gt_occupancy(r.dataset, r.case, r.side)
        return {"ap": ap, "lat": lat, "gt": gt,
                "dataset": r.dataset, "case": r.case, "side": r.side, "variant": r.variant}

# ---- knee-level K-FOLD cross-validation split (dataset-stratified) ----
# Replaces the single 70/15/15 split. With only ~13 fractured knees a single test split holds ~2
# fractured cases, far too few for a credible conclusion (see RSNA Radiology:AI 2022 small-sample
# guidance). 5-fold CV (as in FracReconNet, PMC9829664) puts every knee in a test fold exactly once,
# so all 13 fractured knees get tested across the run. Round-robin slicing keeps fold sizes balanced;
# test = fold k, val = fold (k+1), train = the rest. The split is over (case, side) so every variant
# of a knee shares a fold (no leakage). Loading the per-fold CSV written by frontend_pretrain.ipynb
# guarantees identical fold boundaries across encoder/frontend/unet/vnet.
def kfold_split(df, n_folds=N_FOLDS, fold=FOLD, seed=SEED):
    rng = random.Random(seed); assign = {}
    for ds, g in df.groupby("dataset"):
        keys = sorted({(r.case, r.side) for r in g.itertuples()})
        rng.shuffle(keys)
        folds = [keys[i::n_folds] for i in range(n_folds)]      # round-robin -> balanced sizes
        test_keys = set(folds[fold % n_folds]); val_keys = set(folds[(fold + 1) % n_folds])
        for k in keys:
            kk = (ds,) + k
            assign[kk] = "test" if k in test_keys else ("val" if k in val_keys else "train")
    return df.apply(lambda r: assign[(r.dataset, r.case, r.side)], axis=1)

if SMOKE_TEST:
    # tiny, fast subset just to verify the pipeline runs end-to-end (keep FOLD=0 in smoke)
    keep = paired_index[paired_index.variant == "normal"]
    sub = [g[g.case.isin(list(dict.fromkeys(g.case))[:SMOKE_CASES_PER_GROUP])]
           for _, g in keep.groupby("dataset")]
    paired_index = pd.concat(sub).reset_index(drop=True)

split_csv = MODELS_DIR / "decoders" / ("decoder_split_fold%d.csv" % FOLD)
split_csv.parent.mkdir(parents=True, exist_ok=True)
if split_csv.exists():
    # Load the exact per-fold split written by frontend_pretrain.ipynb to guarantee identical
    # train/val/test assignment across all notebooks (frontend + unet + vnet) for this FOLD.
    saved = pd.read_csv(split_csv)[["dataset", "case", "side", "split"]].drop_duplicates(
        subset=["dataset", "case", "side"])
    paired_index = paired_index.merge(saved, on=["dataset", "case", "side"], how="left")
    n_missing = paired_index["split"].isna().sum()
    if n_missing:
        print("[warn] %d rows not in split CSV; assigning to train. "
              "Re-run frontend_pretrain.ipynb (same FOLD) on the full dataset first." % n_missing)
        paired_index["split"] = paired_index["split"].fillna("train")
    print("loaded fold %d split from %s" % (FOLD, split_csv.name))
else:
    paired_index["split"] = kfold_split(paired_index)
    paired_index[["dataset", "case", "side", "variant", "split"]].to_csv(split_csv, index=False)
    print("[warn] split CSV not found; recomputed fold %d and saved. "
          "Run frontend_pretrain.ipynb (same FOLD) first for a guaranteed-identical split." % FOLD)
print(paired_index.groupby(["split", "dataset"]).size())

In [ ]:
train_df = paired_index[paired_index.split == "train"]
val_df   = paired_index[paired_index.split == "val"]
test_df  = paired_index[paired_index.split == "test"]

def make_loader(df, shuffle):
    if len(df) == 0:
        return None
    return DataLoader(PairedDRRVolumeDataset(df), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, drop_last=False)

train_loader = make_loader(train_df, True)
val_loader   = make_loader(val_df, False)
test_loader  = make_loader(test_df, False)
print("samples -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

## 3. The decoders — the ONLY difference between the two models

Both decoders use the **same wiring**: they take the encoder's 4 feature levels, upsample the
spatial size step by step (`8 -> 16 -> 32 -> 64`) while concatenating the matching encoder feature
as a **skip connection** (classic U-Net shape), then a **super-resolution head** grows the small
`(LIFT_DEPTH, 64, 64)` grid up to the full `TARGET_RES^3` output. Channels taper at high resolution
to keep memory affordable.

The single difference is the **building block**:
- **`unet`** -> `DoubleConv`: two `Conv3d -> BatchNorm -> ReLU`. Plain, no residual.
- **`vnet`** -> `VNetResBlock`: two `Conv3d -> BatchNorm -> PReLU` **plus a residual add** (V-Net's
  signature). Residual connections help gradients flow in deep volumetric nets.

Because everything else (encoder, skips, resolution, loss) is identical, any score difference is
attributable to the decoder design.

> **Naming precision (state this in the write-up):** this isolates the *decoder conv-block* (plain
> `DoubleConv` vs. residual `VNetResBlock`) inside a *shared* ConvNeXtV2 encoder–decoder. It is not
> canonical "3D U-Net vs V-Net" — V-Net's learned downsampling and its Dice objective live in the
> shared parts. Report it as a controlled decoder-block ablation, not an architecture comparison.

> **Known confound (documented):** the encoder fixes the lift depth at `LIFT_DEPTH` (16). The z-axis
> is grown to `TARGET_RES` only by trilinear interpolation inside the super-resolution head, which
> cannot synthesise fracture-line detail along depth. This fixed, *shared* bottleneck likely
> dominates over the U-Net/V-Net block difference, and is the prime candidate for the *conditional*
> fracture-aware stretch goal.

In [ ]:
def conv_block(block_type, in_ch, out_ch):
    return DoubleConv(in_ch, out_ch) if block_type == "unet" else VNetResBlock(in_ch, out_ch)

class DoubleConv(nn.Module):
    """U-Net block: (Conv3d -> BN -> ReLU) x2. Plain, no residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class VNetResBlock(nn.Module):
    """V-Net block: (Conv3d -> BN -> PReLU) x2 + residual add (input projected if channels differ)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1); self.n1 = nn.BatchNorm3d(out_ch); self.a1 = nn.PReLU(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1); self.n2 = nn.BatchNorm3d(out_ch); self.a2 = nn.PReLU(out_ch)
    def forward(self, x):
        y = self.a1(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return self.a2(y + self.proj(x))

class SuperResHead(nn.Module):
    """Grow the (LIFT_DEPTH, 64, 64) feature grid up to (T,T,T) via staged trilinear upsample +
    refine blocks with tapering channels (heavy work stays at low resolution -> low memory)."""
    def __init__(self, block_type, in_ch, target, use_grad_ckpt=False):
        super().__init__()
        self.target = tuple(int(t) for t in target); self.use_grad_ckpt = use_grad_ckpt
        self.b1 = conv_block(block_type, in_ch, 32)
        self.b2 = conv_block(block_type, 32, 16)
        self.b3 = conv_block(block_type, 16, 8)
        self.out = nn.Conv3d(8, 1, 1)
    def _run(self, blk, x):
        if self.use_grad_ckpt and x.requires_grad:
            return cp.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):
        d0, h0, w0 = x.shape[-3:]; dt, ht, wt = self.target
        s1 = (round(d0 + (dt - d0) / 3), round(h0 + (ht - h0) / 3), round(w0 + (wt - w0) / 3))
        s2 = (round(d0 + 2 * (dt - d0) / 3), round(h0 + 2 * (ht - h0) / 3), round(w0 + 2 * (wt - w0) / 3))
        x = F.interpolate(x, size=s1, mode="trilinear", align_corners=False); x = self._run(self.b1, x)
        x = F.interpolate(x, size=s2, mode="trilinear", align_corners=False); x = self._run(self.b2, x)
        x = F.interpolate(x, size=self.target, mode="trilinear", align_corners=False); x = self._run(self.b3, x)
        return self.out(x)

class Decoder3D(nn.Module):
    """Multi-scale skip-connected decoder. Same wiring for both models; only the block differs."""
    def __init__(self, block_type, enc_channels=OUT_CHANNELS, target=(64, 64, 64),
                 use_grad_ckpt=False, deep_supervision=False):
        super().__init__()
        c0, c1, c2, c3 = enc_channels
        self.deep_supervision = deep_supervision; self.target = tuple(int(t) for t in target)
        self.up3 = nn.ConvTranspose3d(c3, c2, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec3 = conv_block(block_type, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec2 = conv_block(block_type, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        self.dec1 = conv_block(block_type, c0 + c0, c0)
        self.sr = SuperResHead(block_type, c0, self.target, use_grad_ckpt)
        if deep_supervision:
            self.aux3 = nn.Conv3d(c2, 1, 1); self.aux2 = nn.Conv3d(c1, 1, 1); self.aux1 = nn.Conv3d(c0, 1, 1)
    def forward(self, feats):
        l0, l1, l2, l3 = feats
        x = self.up3(l3); x = torch.cat([x, l2], 1); x = self.dec3(x); a3 = x
        x = self.up2(x);  x = torch.cat([x, l1], 1); x = self.dec2(x); a2 = x
        x = self.up1(x);  x = torch.cat([x, l0], 1); x = self.dec1(x); a1 = x
        out = self.sr(x)
        if self.deep_supervision and self.training:
            up = lambda h: F.interpolate(h, size=self.target, mode="trilinear", align_corners=False)
            return out, [up(self.aux3(a3)), up(self.aux2(a2)), up(self.aux1(a1))]
        return out, None

class ReconModel(nn.Module):
    """Full model = shared bi-planar encoder/fusion + a (U-Net or V-Net) decoder."""
    def __init__(self, fusion, decoder):
        super().__init__(); self.fusion = fusion; self.decoder = decoder
    def forward(self, ap, lat):
        _, f3d = self.fusion(ap, lat)
        return self.decoder(f3d)

In [ ]:
def build_model():
    # The fusion (encoder + bi-planar fusion + 2D->3D lift) is built once; how we initialise and
    # freeze it depends on the comparison mode.
    fusion = BiPlanarFeatureFusion(depth=LIFT_DEPTH, pretrained=PRETRAINED,
                                   freeze_encoder=(FREEZE_ENCODER and not FREEZE_FRONTEND))
    if FREEZE_FRONTEND and FRONTEND_CKPT.exists():
        # STRICT comparison: load the pretrained front-end and freeze it WHOLESALE, so both the
        # U-Net and V-Net runs consume byte-identical features (only the decoder differs).
        sd = torch.load(FRONTEND_CKPT, map_location="cpu")["front_end"]
        missing, unexpected = fusion.load_state_dict(sd, strict=False)
        for p in fusion.parameters():
            p.requires_grad = False
        print("loaded FROZEN front-end from %s: missing=%d unexpected=%d"
              % (FRONTEND_CKPT.name, len(missing), len(unexpected)))
    elif SIMCLR_CKPT.exists():
        # FALLBACK: only the SimCLR encoder is pretrained; fusion+lift train with the decoder.
        if FREEZE_FRONTEND:
            print("[warn] FREEZE_FRONTEND=True but %s not found; run frontend_pretrain.ipynb first. "
                  "Falling back to SimCLR-encoder-only." % FRONTEND_CKPT.name)
        fusion.load_simclr_encoder(SIMCLR_CKPT)
    else:
        print("[warn] no front-end or SimCLR checkpoint; encoder uses ImageNet/random init.")
    decoder = Decoder3D(MODEL, target=(TARGET_RES,) * 3,
                        use_grad_ckpt=USE_GRAD_CKPT, deep_supervision=DEEP_SUPERVISION)
    return ReconModel(fusion, decoder)

# shape sanity check (eval mode, no grad -> cheap)
_m = build_model().to(DEVICE).eval()
with torch.no_grad():
    _ap = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    _out, _ = _m(_ap, _ap)
n_train = sum(p.numel() for p in _m.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in _m.parameters() if not p.requires_grad)
print("model:", MODEL, "| output volume:", tuple(_out.shape),
      "| expected:", (1, 1, TARGET_RES, TARGET_RES, TARGET_RES))
print("trainable params: %.2fM (decoder only when FREEZE_FRONTEND) | frozen: %.2fM"
      % (n_train / 1e6, n_frozen / 1e6))
del _m, _ap, _out

In [ ]:
# --- NaN diagnostic (optional): finds the first non-finite tensor. Set False to skip. ---
# If training ever prints NaN, run this: it shows whether the inputs, the fused 3D features, or the
# logits become non-finite, and whether float16 autocast (vs fp32) is what introduces it.
RUN_NAN_DIAGNOSTIC = True
if RUN_NAN_DIAGNOSTIC and train_loader is not None:
    _dm = build_model().to(DEVICE).eval()
    _b = next(iter(train_loader))
    _ap, _lat, _gt = _b["ap"].to(DEVICE), _b["lat"].to(DEVICE), _b["gt"].to(DEVICE)
    def _chk(name, t):
        print("  %-12s finite=%s min=%.3g max=%.3g"
              % (name, bool(torch.isfinite(t).all()), float(t.min()), float(t.max())))
    print("inputs:"); _chk("ap", _ap); _chk("lat", _lat); _chk("gt", _gt)
    for use in ([False, True] if DEVICE.type == "cuda" else [False]):
        ad = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        with torch.no_grad():
            if use:
                with torch.amp.autocast("cuda", dtype=ad):
                    _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
            else:
                _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
        print("autocast", use, "| dtype", (str(ad) if use else "fp32"))
        for i, f in enumerate(_f3):
            _chk("feat3d[%d]" % i, f)
        _chk("logits", _o)
    del _dm, _b, _ap, _lat, _gt

## 4. Loss and metrics

- **Loss = 0.5 * BCE + 0.5 * soft-Dice.** Bone is a small fraction of the volume (~3%), so pure BCE
  is dominated by easy background voxels. Adding Dice directly optimises overlap and handles the
  class imbalance. (Lai's reference used Dice only; adding BCE stabilises early training.)
- **Overlap metrics:** **Dice** and **IoU** on the binarised prediction.
- **Surface metrics (mm):** **HD95** (95th-percentile symmetric surface distance) and **ASSD**
  (average symmetric surface distance). Dice/IoU "evaluate the global regional similarity and may
  neglect the local boundary/surface" ([surface-supervision, arXiv:2405.01204](https://arxiv.org/html/2405.01204v1)),
  but a fracture *is* a boundary phenomenon — so these distance metrics, which
  [FracReconNet (PMC9829664)](https://pmc.ncbi.nlm.nih.gov/articles/PMC9829664/) reports in mm, are
  the ones that actually move for fractures. We convert voxels→mm with `VOXEL_MM` so numbers are
  resolution-independent and comparable to published work.
- All metrics are reported **per knee**, aggregated **mean±std split by healthy vs fractured**, and
  written to a per-knee CSV. The paired U-Net vs V-Net significance test lives in
  `decoder_comparison.ipynb`.

We deliberately implement these by hand (instead of pulling in MONAI) so the formulas are visible
and the notebook runs with the libraries already installed.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, w_bce=0.5, w_dice=0.5, smooth=1.0):
        super().__init__(); self.w_bce = w_bce; self.w_dice = w_dice; self.smooth = smooth
    def _dice(self, logits, target):
        # IMPORTANT: reduce in float32. Under AMP the logits are float16, and at 256^3 a
        # float16 sum of ~16.8M voxels overflows (>65504) -> inf -> Dice becomes NaN.
        p = torch.sigmoid(logits.float()).reshape(logits.size(0), -1)
        t = target.float().reshape(target.size(0), -1)
        inter = (p * t).sum(1); union = p.sum(1) + t.sum(1)
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
    def forward(self, logits, target):
        logits = logits.float()   # compute the loss in fp32 even when the forward pass ran in fp16
        return self.w_bce * F.binary_cross_entropy_with_logits(logits, target.float()) + self.w_dice * self._dice(logits, target)

DICE_BCE = DiceBCELoss()

def total_loss(output, target, aux_weight=0.3):
    out, aux = output
    loss = DICE_BCE(out, target)
    if aux:
        for a in aux:
            loss = loss + aux_weight * DICE_BCE(a, target)
    return loss

@torch.no_grad()
def dice_iou(logits, target, thr=0.5):
    p = (torch.sigmoid(logits.float()) > thr).float().reshape(logits.size(0), -1)
    t = (target > 0.5).float().reshape(target.size(0), -1)
    inter = (p * t).sum(1); psum = p.sum(1); tsum = t.sum(1)
    dice = (2 * inter + 1e-6) / (psum + tsum + 1e-6)
    iou = (inter + 1e-6) / (psum + tsum - inter + 1e-6)
    return dice.cpu().numpy(), iou.cpu().numpy()

# ---- surface (boundary) metrics, reported in MILLIMETRES ----
# Volumetric Dice/IoU "evaluate the global regional similarity and may neglect the local
# boundary/surface" (Cross-Scale Attention & Surface Supervision, arXiv:2405.01204); fractures ARE
# a boundary phenomenon, so we add the two distance metrics the fracture literature reports:
#   HD95 - 95th-percentile symmetric surface distance (localized disagreement)
#   ASSD - average symmetric surface distance (FracReconNet, PMC9829664, reports this in mm)
# VOXEL_MM converts voxel distances to mm using the GT field-of-view, so numbers are resolution- and
# device-independent and comparable to published work.
def _surface_dists(pred_bin, gt_bin, spacing):
    """Returns (pred-surface->GT distances, GT-surface->pred distances) in mm, or None if a mask is empty."""
    from scipy.ndimage import binary_erosion, distance_transform_edt
    sp = pred_bin & ~binary_erosion(pred_bin); sg = gt_bin & ~binary_erosion(gt_bin)
    if sp.sum() == 0 or sg.sum() == 0:
        return None
    dg = distance_transform_edt(~sg) * spacing   # distance of every voxel to the GT surface
    dp = distance_transform_edt(~sp) * spacing   # distance of every voxel to the pred surface
    return dg[sp], dp[sg]                         # symmetric pair

def hd95(pred_bin, gt_bin, spacing=1.0):
    d = _surface_dists(pred_bin, gt_bin, spacing)
    return float("nan") if d is None else float(np.percentile(np.concatenate(d), 95))

def assd(pred_bin, gt_bin, spacing=1.0):
    d = _surface_dists(pred_bin, gt_bin, spacing)
    return float("nan") if d is None else float(np.concatenate(d).mean())

# mm per voxel at TARGET_RES: the predrr CT covers a fixed knee field-of-view, so resampling to
# TARGET_RES^3 preserves the extent and rescales the spacing. Read it from a real GT affine; fall
# back to the confirmed 0.78125 mm @256 (=> 0.78125 * 256 / TARGET_RES) if the file is unavailable.
def _voxel_mm():
    try:
        r = paired_index.iloc[0]
        img = nib.load(str(gt_file(r.dataset, r.case, r.side)))
        zooms = np.asarray(img.header.get_zooms()[:3], dtype=float)
        dims = np.asarray(img.shape[:3], dtype=float)
        return float(np.mean(zooms * dims / TARGET_RES))   # extent_mm / TARGET_RES, averaged over axes
    except Exception as e:
        print("[warn] could not read GT spacing (%s); using 0.78125mm@256 fallback." % type(e).__name__)
        return 0.78125 * 256.0 / TARGET_RES
VOXEL_MM = _voxel_mm()
print("voxel spacing for surface metrics: %.4f mm" % VOXEL_MM)

In [ ]:
def run_epoch(model, loader, optimizer, scaler, train):
    model.train(train)
    if FREEZE_FRONTEND:
        model.fusion.eval()   # frozen front-end stays in eval: disables timm drop_path so the
                              # features are deterministic & identical across the U-Net/V-Net runs
    use_amp = USE_AMP and DEVICE.type == "cuda"
    # bfloat16 has float32's dynamic range -> no overflow at 65504 (the float16 NaN cause).
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    use_scaler = use_amp and amp_dtype == torch.float16   # only float16 needs GradScaler
    tot, n, steps, skipped = 0.0, 0, 0, 0
    for batch in loader:
        ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE); gt = batch["gt"].to(DEVICE)
        with torch.set_grad_enabled(train):
            if use_amp:
                with torch.amp.autocast("cuda", dtype=amp_dtype):
                    loss = total_loss(model(ap, lat), gt)
            else:
                loss = total_loss(model(ap, lat), gt)
        if not torch.isfinite(loss):                       # never backprop a NaN/inf loss
            skipped += 1; optimizer.zero_grad(set_to_none=True); continue
        if train:
            optimizer.zero_grad(set_to_none=True)
            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                prev = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= prev: steps += 1   # scaler did not skip the step
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); steps += 1
        tot += loss.item() * ap.size(0); n += ap.size(0)
    if skipped: print("  [warn] skipped %d non-finite batch(es)" % skipped)
    return tot / max(n, 1), steps

@torch.no_grad()
def evaluate(model, loader, surface=False):
    """Per-knee Dice/IoU (always) plus HD95/ASSD in mm (when surface=True). Returns the per-knee
    DataFrame, the overall mean, and a per-group (healthy vs fractured) mean+std table. Surface
    metrics are skipped during per-epoch validation (slow) and computed only for the final test."""
    model.eval(); rows = []
    for batch in loader:
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
        d, i = dice_iou(out, batch["gt"].to(DEVICE))
        if surface:
            prob = torch.sigmoid(out.float()).cpu().numpy(); gtn = batch["gt"].numpy()
        for b in range(len(d)):
            rec = dict(dataset=batch["dataset"][b], case=batch["case"][b],
                       side=batch["side"][b], dice=float(d[b]), iou=float(i[b]))
            if surface:
                pb = prob[b, 0] > 0.5; gb = gtn[b, 0] > 0.5
                rec["hd95_mm"] = hd95(pb, gb, VOXEL_MM)
                rec["assd_mm"] = assd(pb, gb, VOXEL_MM)
            rows.append(rec)
    df = pd.DataFrame(rows)
    cols = [c for c in ["dice", "iou", "hd95_mm", "assd_mm"] if c in df.columns]
    overall = df[cols].mean().to_dict() if len(df) else {c: float("nan") for c in cols}
    by = df.groupby("dataset")[cols].agg(["mean", "std"]) if len(df) else None
    return df, overall, by

def save_ckpt(path, model, optimizer, scheduler, epoch, val_metrics):
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict() if scheduler else None, "val_metrics": val_metrics,
                "config": {"MODEL": MODEL, "FOLD": FOLD, "N_FOLDS": N_FOLDS, "REGIME": REGIME,
                           "TARGET_RES": TARGET_RES, "LIFT_DEPTH": LIFT_DEPTH,
                           "GT_THRESH": GT_THRESH, "FROZEN_FRONTEND": FREEZE_FRONTEND}}, path)

## 5. Training with checkpointing

Each epoch we train, validate, score Dice/IoU on the validation set, step the cosine LR schedule,
and **save checkpoints**:
- `MODEL_last.pth` — always the most recent (for resuming),
- `MODEL_epochNNN.pth` — every `CKPT_EVERY` epochs (so you can **revisit any epoch** later),
- `MODEL_best.pth` — whenever validation Dice improves,
- `MODEL_history.csv` — per-epoch losses/metrics for the learning-curve plot.

Each checkpoint stores the epoch, model + optimizer + scheduler state, the validation metrics, and
the run config (model type, resolution, lift depth, GT threshold) — everything needed to resume or
to load the model later in the UI. On GPU we use **AMP** (mixed precision) and optional **gradient
checkpointing** to fit `256^3` in memory.

In [ ]:
model = build_model().to(DEVICE)
# when FREEZE_FRONTEND, the whole front-end is frozen -> optimiser holds only the decoder's params
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

start_epoch, best_dice, history = 0, -1.0, []
if RESUME_FROM:
    ck = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    if ck.get("scheduler"):
        scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1
    print("resumed from %s at epoch %d" % (RESUME_FROM, start_epoch))

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    tr, steps = run_epoch(model, train_loader, optimizer, scaler, train=True)
    if val_loader:
        va, _ = run_epoch(model, val_loader, optimizer, scaler, train=False)
        _, overall, _ = evaluate(model, val_loader)
    else:
        va, overall = float("nan"), {"dice": float("nan"), "iou": float("nan")}
    if steps > 0:                 # optimizer.step() ran this epoch -> correct order to step scheduler
        scheduler.step()
    history.append(dict(epoch=epoch, train_loss=tr, val_loss=va, val_dice=overall["dice"],
                        val_iou=overall["iou"], lr=optimizer.param_groups[0]["lr"],
                        secs=round(time.time() - t0, 1)))
    pd.DataFrame(history).to_csv(CKPT_DIR / ("%s_history.csv" % MODEL), index=False)
    save_ckpt(CKPT_DIR / ("%s_last.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    if epoch % CKPT_EVERY == 0:
        save_ckpt(CKPT_DIR / ("%s_epoch%03d.pth" % (MODEL, epoch)), model, optimizer, scheduler, epoch, overall)
    if overall["dice"] > best_dice:
        best_dice = overall["dice"]
        save_ckpt(CKPT_DIR / ("%s_best.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    print("epoch %03d | train %.4f | val %.4f | val_dice %.4f | best %.4f"
          % (epoch, tr, va, overall["dice"], best_dice))
print("done. checkpoints in", CKPT_DIR)

## 6. Final evaluation (overall + healthy vs fractured)

We reload the **best** checkpoint and score it on the held-out **test fold** (falls back to the
validation split for the smoke test). This writes the **per-knee metric CSV**
(`<MODEL>_test_metrics.csv`) for this `fold`/`regime`/`model`. The full comparison is assembled by
running every fold for `MODEL="unet"` and `MODEL="vnet"` under both `REGIME="frozen"` and
`REGIME="finetuned"`, then opening **`decoder_comparison.ipynb`**, which pools all the per-knee CSVs
and runs the paired **Wilcoxon signed-rank** test (U-Net vs V-Net) per metric, overall and within
the fractured subgroup.

In [ ]:
best_path = CKPT_DIR / ("%s_best.pth" % MODEL)
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
    print("loaded", best_path.name)

# Final test scoring: Dice/IoU + HD95/ASSD (mm) per knee, aggregated mean+std by group, and the
# per-knee CSV that decoder_comparison.ipynb reads to run the paired U-Net vs V-Net Wilcoxon test.
# (Falls back to the val split for the smoke test, which may have no test cases.)
eval_loader = test_loader or val_loader
if eval_loader:
    df, overall, by = evaluate(model, eval_loader, surface=True)
    df.insert(0, "model", MODEL); df.insert(1, "fold", FOLD); df.insert(2, "regime", REGIME)
    metrics_csv = CKPT_DIR / ("%s_test_metrics.csv" % MODEL)
    df.to_csv(metrics_csv, index=False)
    print("OVERALL:", {k: round(v, 4) for k, v in overall.items()})
    if by is not None:
        print("\nBY GROUP (healthy vs fractured) - mean/std:\n", by.round(4))
    print("\nwrote per-knee metrics ->", metrics_csv)
else:
    print("no eval data in this (smoke) split.")

In [ ]:
h = pd.read_csv(CKPT_DIR / ("%s_history.csv" % MODEL))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("%s loss" % MODEL); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h.epoch, h.val_dice, label="val Dice"); ax[1].plot(h.epoch, h.val_iou, label="val IoU")
ax[1].set_title("%s val metrics" % MODEL); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Quality-assurance views

**(A) Tune `GT_THRESH`.** The first row shows the windowed CT slice and the bone mask at a few
thresholds with the resulting occupancy fraction. Pick a threshold that isolates bone (a few % of
voxels) without swallowing soft tissue, set it in the CONFIG cell, and **delete the
`predrr_occupancy_*` cache folder** so it re-binarises.

**(B) Reconstruction preview.** Mid-slices of the prediction next to the GT, as a quick visual
sanity check (it will look rough after a 2-epoch smoke run — that is expected).

In [ ]:
# (A) GT bone-occupancy threshold QA
sample = paired_index.iloc[0]
vol = nib.load(str(gt_file(sample.dataset, sample.case, sample.side))).get_fdata().astype(np.float32)
mid = vol.shape[2] // 2
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
ax[0].imshow(vol[:, :, mid], cmap="gray"); ax[0].set_title("CT (windowed)")
for j, thr in enumerate([0.3, 0.4, 0.5]):
    ax[j + 1].imshow(vol[:, :, mid] > thr, cmap="gray")
    ax[j + 1].set_title("occ>%.1f  (%.1f%%)" % (thr, (vol > thr).mean() * 100))
for a in ax:
    a.axis("off")
plt.suptitle("%s %s %s - current GT_THRESH=%.2f" % (sample.dataset, sample.case, sample.side, GT_THRESH))
plt.tight_layout(); plt.show()

# (B) reconstruction preview vs GT
if eval_loader:
    batch = next(iter(eval_loader))
    with torch.no_grad():
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
    pred = (torch.sigmoid(out[0, 0]).cpu().numpy() > 0.5).astype(float)
    gtv = batch["gt"][0, 0].numpy()
    m = pred.shape[0] // 2
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    ax[0].imshow(batch["ap"][0, 0], cmap="gray"); ax[0].set_title("input AP")
    ax[1].imshow(gtv[m], cmap="gray"); ax[1].set_title("GT mid-slice")
    ax[2].imshow(pred[m], cmap="gray"); ax[2].set_title("prediction mid-slice")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

## Next steps

- Train `MODEL = "unet"` to convergence, then re-run with `MODEL = "vnet"`.
- Compare the two `*_history.csv` files and the per-group test tables (healthy vs fractured).
- Copy the `models/decoders/` checkpoints back to your machine and explore them in `decoder_ui.ipynb`.
- If `256^3` runs out of GPU memory: keep `BATCH_SIZE = 1`, ensure `USE_AMP` and `USE_GRAD_CKPT`
  are `True`, or temporarily set `FREEZE_ENCODER = True` to cut activation memory.